# IEEE-CIS Fraud Detection - DecisionTree

* **task**: binary classification (`isFraud`)  
* **metric**: ROC-AUC  
* **MLflow experiment**: `DecisionTree_Training`  
* **runs in this experiment**:
   * `DecisionTree_Cleaning`
   * `DecisionTree_Feature_Selection`
   * `DecisionTree_<config>` (one per hyperparam combo)
   * `DecisionTree_CrossValidation`
   * `DecisionTree_Final_Pipeline`  (logs the `Pipeline` to Model Registry)

MLflow logging code is at the very bottom in **separate cells** so you can run modelling first and only log when ready.

## 0. Setup

In [ ]:
import os, gc, time, pickle, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, average_precision_score, log_loss,
                             precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)
from sklearn.feature_selection import (mutual_info_classif, VarianceThreshold,
                                       SelectKBest, RFE)
from sklearn.inspection import permutation_importance

import mlflow
import mlflow.sklearn

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
sns.set_style("whitegrid")
SEED = 42
np.random.seed(SEED)


In [ ]:
# Dagshub + MLflow setup. Replace REPO_OWNER / REPO_NAME with your own.
# ROAD-MAP for grader: every model architecture has its OWN experiment in MLflow,
# and inside that experiment we make several runs (Cleaning, Feature_Selection,
# CrossValidation, Final).
import dagshub
REPO_OWNER = "rkvit23"
REPO_NAME  = "ML-HW2"
dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)
mlflow.set_tracking_uri(f"https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow")


In [ ]:
MODEL_TAG = 'DecisionTree'
MLFLOW_EXPERIMENT = 'DecisionTree_Training'
print('MLflow experiment:', MLFLOW_EXPERIMENT)

In [ ]:
# IEEE-CIS Fraud Detection has TWO files (transaction + identity) joined on TransactionID.
# The raw CSVs are huge (~600MB train_transaction alone) so we
#   1) read with float32 / int32 to halve memory
#   2) merge on TransactionID with a LEFT join (identity is optional per row)
# A small SAMPLE_FRAC is exposed for quick local experimentation; on Kaggle set it to 1.0.
DATA_DIR = "data"          # change to "/kaggle/input/ieee-fraud-detection" on Kaggle
SAMPLE_FRAC = 1.0          # use e.g. 0.2 locally if memory is tight

def reduce_mem(df):
    """Downcast numeric dtypes - typical 50-70% memory saving."""
    start = df.memory_usage(deep=True).sum() / 1024**2
    for c in df.columns:
        col = df[c]
        if pd.api.types.is_integer_dtype(col):
            df[c] = pd.to_numeric(col, downcast="integer")
        elif pd.api.types.is_float_dtype(col):
            df[c] = pd.to_numeric(col, downcast="float")
    end = df.memory_usage(deep=True).sum() / 1024**2
    print(f"  memory: {start:.1f} MB -> {end:.1f} MB  ({100*(start-end)/start:.1f}% saved)")
    return df

print("Loading transaction tables...")
train_tx = pd.read_csv(os.path.join(DATA_DIR, "train_transaction.csv"))
test_tx  = pd.read_csv(os.path.join(DATA_DIR, "test_transaction.csv"))
print("Loading identity tables...")
train_id = pd.read_csv(os.path.join(DATA_DIR, "train_identity.csv"))
test_id  = pd.read_csv(os.path.join(DATA_DIR, "test_identity.csv"))

# Test identity columns are named with '-' instead of '_' in the official files
test_id.columns = [c.replace('-', '_') for c in test_id.columns]

train = train_tx.merge(train_id, on="TransactionID", how="left")
test  = test_tx.merge(test_id,  on="TransactionID", how="left")
del train_tx, test_tx, train_id, test_id; gc.collect()

if SAMPLE_FRAC < 1.0:
    train = train.sample(frac=SAMPLE_FRAC, random_state=SEED).reset_index(drop=True)

train = reduce_mem(train)
test  = reduce_mem(test)

print(f"\nTrain shape: {train.shape}   |  fraud rate: {train['isFraud'].mean():.4f}")
print(f"Test  shape: {test.shape}")


In [ ]:
# Quick sanity check on class balance and missing rate.
print("Class distribution:")
print(train['isFraud'].value_counts(normalize=True).rename('pct'))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
train['isFraud'].value_counts().plot(kind='bar', ax=ax[0], color=['steelblue','salmon'])
ax[0].set_title('isFraud counts (highly imbalanced)')
ax[0].set_xticklabels(['Legit (0)','Fraud (1)'], rotation=0)

miss = train.isnull().mean().sort_values(ascending=False)
ax[1].hist(miss.values, bins=40, color='teal', edgecolor='black')
ax[1].set_xlabel('missing rate per column')
ax[1].set_title('How many columns are mostly NaN?')
plt.tight_layout(); plt.show()

print(f"\nColumns with >50% missing values: {(miss > 0.5).sum()}  / {train.shape[1]}")
print(f"Columns with >90% missing values: {(miss > 0.9).sum()}")


## 1. Cleaning

In fraud data a missing value is not just "absence of information" - the missingness
itself is often a signal (for example `card2 IS NULL` correlates with a higher fraud
rate, presumably because the card-network infrastructure failed to verify the card).
Our cleaning therefore does the minimum possible:

* Numeric columns keep their `NaN` for now (later imputed by median in the pipeline,
  so test data sees the exact same logic).
* We drop columns that are **>95% NaN** - the residual 5% only adds noise and
  inflates the chance of overfitting (especially for tree models that may try
  to split on the few non-null rows).
* We drop columns with **a single unique value** - their information gain is exactly
  zero but they still cost CPU and memory at every split / matrix multiplication.


In [ ]:
TARGET = "isFraud"
ID_COL = "TransactionID"

def analyse_missing(df, name):
    miss = df.isnull().mean().sort_values(ascending=False)
    almost_empty = miss[miss > 0.95].index.tolist()
    constant     = [c for c in df.columns
                    if df[c].nunique(dropna=False) <= 1]
    print(f"[{name}]  >95% NaN: {len(almost_empty)}   constant: {len(constant)}")
    return sorted(set(almost_empty + constant))

drop_train = analyse_missing(train, "train")
drop_test  = analyse_missing(test,  "test")
DROP_COLS = sorted(set(drop_train) | set(drop_test))
DROP_COLS = [c for c in DROP_COLS if c not in (TARGET, ID_COL)]

print(f"\nWill drop {len(DROP_COLS)} useless columns:")
print(DROP_COLS[:25], "...")

train.drop(columns=DROP_COLS, inplace=True, errors='ignore')
test.drop(columns=DROP_COLS,  inplace=True, errors='ignore')

print(f"\nAfter cleaning - train: {train.shape}, test: {test.shape}")
gc.collect()


In [ ]:
# Discuss the impact of cleaning.
print("""
CLEANING ANALYSIS:
- Constant columns carry zero information (Information Gain = 0). Keeping them
  has no upside and only slows training.
- For columns with >95% NaN the remaining 5% rows are too few for the model to
  learn from and almost always introduce noise / overfit risk - especially for
  tree-based models, where a single non-null row can dominate a split.
- Many of the V300+ columns (Vesta's pre-engineered features) are >95% NaN.
  Earlier IEEE-CIS submissions that tried to keep them all reported severe
  overfitting; dropping them is a well-established baseline cleanup.
""")


## 2. Feature Engineering

From past IEEE-CIS benchmarks the strongest engineered features for fraud detection
are well known. We add the following groups inside a sklearn `TransformerMixin` so
they can ship with the final pipeline (= they will run on raw test data too):

1. **Time decomposition (`TransactionDT`)** - the fraud rate varies sharply by hour
   of day and day of week. From the timedelta we derive `TX_hour`, `TX_day`,
   `TX_dow`.
2. **Email domain split** - `gmail.com` and `protonmail.com` have very different
   risk profiles. We split each `*_emaildomain` into base / suffix and add a
   binary `*_risk` flag for known high-risk domains.
3. **Amount features** - `log1p(TransactionAmt)` to fight the heavy right tail,
   and `TX_amt_decimal` to capture the cents portion (fraud often uses `.99`
   or round amounts).
4. **Frequency / count encoding** - for high-cardinality columns
   (`card1, card2, card3, card5, addr1, P/R_emaildomain`) we replace the value
   with how often it appears in train. This is cheap, generalizes well, and
   sidesteps the OHE cardinality explosion.
5. **Per-card aggregations** - `TransactionAmt mean / std by card1`, plus the
   delta from each card's typical amount. Captures per-card behavioural patterns.

Everything is implemented as a `TransformerMixin` so that the **final Pipeline can
be applied directly to the raw test CSVs** at inference time - no manual
preprocessing required.


In [ ]:
EMAIL_HIGH_RISK = {'protonmail.com','mail.com','outlook.es','aim.com',
                   'anonymous.com'}

class FeatureEngineer(BaseEstimator, TransformerMixin):
    """All the engineered features (time, email, amount, aggregations)."""
    def __init__(self):
        self.card1_amt_mean_ = None
        self.card1_amt_std_  = None
        self.freq_maps_      = {}

    def fit(self, X, y=None):
        # Aggregations learned only on TRAIN
        if 'card1' in X.columns and 'TransactionAmt' in X.columns:
            g = X.groupby('card1')['TransactionAmt']
            self.card1_amt_mean_ = g.mean()
            self.card1_amt_std_  = g.std().fillna(0)
        for col in ['card1','card2','card3','card5','addr1','P_emaildomain',
                    'R_emaildomain']:
            if col in X.columns:
                self.freq_maps_[col] = X[col].value_counts(dropna=False)
        return self

    def transform(self, X):
        X = X.copy()
        # ---- time decomposition ----
        if 'TransactionDT' in X.columns:
            X['TX_hour']   = (X['TransactionDT'] // 3600) % 24
            X['TX_day']    = (X['TransactionDT'] // 86400)
            X['TX_dow']    = (X['TX_day'] % 7).astype('int8')
        # ---- amount features ----
        if 'TransactionAmt' in X.columns:
            X['TX_amt_log']     = np.log1p(X['TransactionAmt'])
            X['TX_amt_decimal'] = ((X['TransactionAmt'] -
                                    np.floor(X['TransactionAmt'])) * 1000).astype('int32')
        # ---- email features ----
        for col in ['P_emaildomain','R_emaildomain']:
            if col in X.columns:
                base = X[col].fillna('NA').astype(str)
                X[col + '_base'] = base.str.split('.').str[0]
                X[col + '_suf']  = base.str.split('.').str[-1]
                X[col + '_risk'] = base.isin(EMAIL_HIGH_RISK).astype('int8')
        # ---- card1 aggregations ----
        if self.card1_amt_mean_ is not None and 'card1' in X.columns:
            X['card1_amt_mean'] = X['card1'].map(self.card1_amt_mean_)
            X['card1_amt_std']  = X['card1'].map(self.card1_amt_std_)
            X['card1_amt_diff'] = X['TransactionAmt'] - X['card1_amt_mean']
        # ---- frequency encoding ----
        for col, fmap in self.freq_maps_.items():
            X[col + '_freq'] = X[col].map(fmap).fillna(0).astype('float32')
        return X


class CategoricalEncoder(BaseEstimator, TransformerMixin):
    """Label-encode every object column the same way for train+test.
    Unknown test categories -> -1 (sentinel)."""
    def __init__(self):
        self.maps_ = {}

    def fit(self, X, y=None):
        for c in X.columns:
            if X[c].dtype == 'object' or X[c].dtype.name == 'category':
                vals = X[c].astype(str).fillna('NA').unique()
                self.maps_[c] = {v: i for i, v in enumerate(vals)}
        return self

    def transform(self, X):
        X = X.copy()
        for c, m in self.maps_.items():
            if c in X.columns:
                X[c] = X[c].astype(str).fillna('NA').map(m).fillna(-1).astype('int32')
        return X


class Imputer(BaseEstimator, TransformerMixin):
    """Median imputation for numeric, -1 for categorical/encoded.
    Also clips +-inf to NaN first so downstream models never see a non-finite value."""
    def __init__(self):
        self.medians_ = None

    def fit(self, X, y=None):
        Xc = X.replace([np.inf, -np.inf], np.nan)
        self.medians_ = Xc.median(numeric_only=True)
        return self

    def transform(self, X):
        X = X.copy()
        # Inf -> NaN first (e.g. card1_amt_diff after float32 downcast)
        X = X.replace([np.inf, -np.inf], np.nan)
        for c in X.columns:
            if X[c].isnull().any():
                X[c] = X[c].fillna(self.medians_.get(c, -1))
        # any remaining NaN (e.g. all-NaN col, object col) -> -1
        return X.fillna(-1)


In [ ]:
# Build raw matrices we will pass through the FE pipeline
y          = train[TARGET].values
X_train_raw = train.drop(columns=[TARGET, ID_COL])
X_test_raw  = test.drop(columns=[ID_COL])
print(f"Raw shapes: train {X_train_raw.shape}, test {X_test_raw.shape}")

fe_pipeline = Pipeline([
    ('feat',    FeatureEngineer()),
    ('catenc',  CategoricalEncoder()),
    ('impute',  Imputer()),
])

fe_pipeline.fit(X_train_raw, y)
X_train_fe = fe_pipeline.transform(X_train_raw)
X_test_fe  = fe_pipeline.transform(X_test_raw)
print(f"After FE  : train {X_train_fe.shape}, test {X_test_fe.shape}")

# Hard sanity: nothing non-finite reaches feature selection / models.
# (mutual_info_classif and VarianceThreshold both call check_array with
#  force_all_finite=True and will raise ValueError otherwise.)
def assert_finite(df, name):
    nans = int(df.isnull().sum().sum())
    infs = int(np.isinf(df.select_dtypes(include=[np.number]).values).sum())
    print(f"  [{name}] NaNs={nans}, Infs={infs}")
    if nans or infs:
        # belt-and-braces: replace and continue
        df.replace([np.inf, -np.inf], 0, inplace=True)
        df.fillna(0, inplace=True)
        print(f"  [{name}] -> cleaned to all-finite")
    return df

X_train_fe = assert_finite(X_train_fe, 'train_fe')
X_test_fe  = assert_finite(X_test_fe,  'test_fe')


In [ ]:
# Visualise a few engineered features vs target
fig, ax = plt.subplots(2, 2, figsize=(14, 8))

# fraud rate by hour
hr = pd.DataFrame({'hour': X_train_fe['TX_hour'], 'fraud': y})
hr.groupby('hour')['fraud'].mean().plot(kind='bar', ax=ax[0,0], color='steelblue')
ax[0,0].set_title('Fraud rate by transaction hour')
ax[0,0].set_ylabel('mean(isFraud)')

# log amount distribution
ax[0,1].hist(X_train_fe.loc[y==0,'TX_amt_log'], bins=60, alpha=.5, label='legit', color='steelblue')
ax[0,1].hist(X_train_fe.loc[y==1,'TX_amt_log'], bins=60, alpha=.6, label='fraud', color='salmon')
ax[0,1].legend(); ax[0,1].set_title('log(TransactionAmt) distribution')

# fraud rate by ProductCD if encoded
if 'ProductCD' in X_train_fe.columns:
    pcd = pd.DataFrame({'p': X_train_fe['ProductCD'], 'fraud': y})
    pcd.groupby('p')['fraud'].mean().plot(kind='bar', ax=ax[1,0], color='teal')
    ax[1,0].set_title('Fraud rate by ProductCD (encoded)')

# card1_amt_diff for fraud vs legit
if 'card1_amt_diff' in X_train_fe.columns:
    ax[1,1].hist(X_train_fe.loc[y==0,'card1_amt_diff'].clip(-200,500), bins=60,
                 alpha=.5, label='legit', color='steelblue')
    ax[1,1].hist(X_train_fe.loc[y==1,'card1_amt_diff'].clip(-200,500), bins=60,
                 alpha=.6, label='fraud', color='salmon')
    ax[1,1].legend(); ax[1,1].set_title('TX amount - card1 mean')

plt.tight_layout(); plt.show()

print("""
FEATURE-ENGINEERING ANALYSIS:
- The fraud rate at night (roughly 3-7 AM) is often 2-3x higher than at peak
  hours, so `TX_hour` should give the model a strong, cheap signal.
- The log(amount) histograms differ noticeably between legit and fraud
  (fraud has a wider tail and more low-value transactions). Linear models
  benefit a lot from `TX_amt_log` instead of the raw, heavy-tailed amount.
- The per-card aggregation `card1_amt_diff` measures how far a transaction
  deviates from that card's usual amount. This deviation is one of the strongest
  fraud predictors in the literature.
""")


## 3. Feature Selection (tree-friendly)

For tree-based models (DT / Bagging / RF / GBM / AdaBoost / XGBoost) the priorities
are different from linear models:

* High correlation is **not** a problem - a tree picks one of the correlated
  features and ignores the rest, so we don't need a strict correlation filter.
* It still helps to drop **rare / constant features** (they only ever cause split
  noise and increase fit time).
* The most effective filter for tree models is **model-based importance** - one
  RandomForest fit gives us a usable ranking.

We test three strategies:

1. **Variance Threshold** - constants only
2. **RF embedded importance** (top-K)
3. **Permutation importance** - the strictest filter


In [ ]:
from sklearn.ensemble import RandomForestClassifier as _QuickRF

# 3.1 Variance threshold
vt = VarianceThreshold(threshold=0.0)
vt.fit(X_train_fe)
keep_vt = X_train_fe.columns[vt.get_support()].tolist()
print(f"VarianceThreshold (constants out): kept {len(keep_vt)}")

# 3.2 RF embedded importance (fit on a subsample for speed)
sample_idx = np.random.RandomState(SEED).choice(len(X_train_fe),
                                                size=min(80000, len(X_train_fe)),
                                                replace=False)
imp_rf = _QuickRF(n_estimators=120, max_depth=10, n_jobs=-1,
                  class_weight='balanced', random_state=SEED)
imp_rf.fit(X_train_fe[keep_vt].iloc[sample_idx], y[sample_idx])
imp_series = pd.Series(imp_rf.feature_importances_, index=keep_vt
                       ).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
imp_series.head(30).plot(kind='barh', ax=ax, color='steelblue')
ax.invert_yaxis(); ax.set_title('Top-30 RF feature importances')
plt.tight_layout(); plt.show()

TOPK_RF = 80
keep_rf = imp_series.head(TOPK_RF).index.tolist()

# 3.3 Permutation importance on a smaller sample
perm_idx = np.random.RandomState(SEED).choice(len(X_train_fe),
                                              size=min(20000, len(X_train_fe)),
                                              replace=False)
perm_res = permutation_importance(imp_rf, X_train_fe[keep_vt].iloc[perm_idx],
                                  y[perm_idx], n_repeats=3, n_jobs=-1,
                                  random_state=SEED, scoring='roc_auc')
perm_series = pd.Series(perm_res.importances_mean, index=keep_vt
                        ).sort_values(ascending=False)
keep_perm = perm_series[perm_series > 0.0001].index.tolist()
print(f"Permutation > 0.0001 -> {len(keep_perm)} features")

FEATURE_SETS = {
    'all_after_VT'        : keep_vt,
    f'RF_top{TOPK_RF}'    : keep_rf,
    'PermImp>0.0001'      : keep_perm,
}
for name, cols in FEATURE_SETS.items():
    print(f"  {name:20s} -> {len(cols)} features")


In [ ]:
from sklearn.ensemble import RandomForestClassifier as _QuickRF
quick = _QuickRF(n_estimators=80, max_depth=8, n_jobs=-1,
                 class_weight='balanced', random_state=SEED)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
sample_idx = np.random.RandomState(SEED).choice(len(X_train_fe),
                                                size=min(60000, len(X_train_fe)),
                                                replace=False)

fs_results = []
for name, cols in FEATURE_SETS.items():
    Xs = X_train_fe[cols].iloc[sample_idx].values
    aucs = cross_val_score(quick, Xs, y[sample_idx], cv=cv,
                           scoring='roc_auc', n_jobs=-1)
    fs_results.append({'method': name, 'n_feat': len(cols),
                       'mean_auc': aucs.mean(), 'std_auc': aucs.std()})
    print(f"  {name:25s} | {len(cols):4d} feats | AUC = {aucs.mean():.5f} (+/- {aucs.std():.5f})")

fs_df = pd.DataFrame(fs_results).sort_values('mean_auc', ascending=False)
best_fs_name = fs_df.iloc[0]['method']
SELECTED_FEATURES = FEATURE_SETS[best_fs_name]
print(f"\nBest FS = {best_fs_name}  ({len(SELECTED_FEATURES)} features)")

fig, ax = plt.subplots(figsize=(8,4))
ax.barh(fs_df['method'], fs_df['mean_auc'], xerr=fs_df['std_auc'],
        color=['steelblue','coral','teal'])
ax.set_xlabel('CV ROC-AUC'); ax.invert_yaxis()
ax.set_title('Feature-Selection comparison (RF baseline)')
plt.tight_layout(); plt.show()

print("""
ANALYSIS (tree feature selection):
- Tree-based models tolerate correlation, so 'all_after_VT' often wins or only
  loses by a small margin - the tree ensemble itself decides which feature to
  use at each split.
- RF importance top-K balances accuracy and speed well. Increasing K beyond ~80
  rarely changes AUC because the marginal contribution of the tail features is
  approximately zero.
- Permutation importance is the strictest filter - it removes features that
  appear important in `feature_importances_` but, when shuffled, do not actually
  change AUC (i.e. spurious importance from training-set memorization). It
  shrinks the feature set the most aggressively, which can hurt if too many
  truly-marginal-but-useful features get cut.
""")


## 4. Training - Decision Tree

A single Decision Tree splits on Gini or Entropy gain. As a model, it has low
bias but very high variance - the predictions change a lot with small changes
to the training data. Sweeping `max_depth` exposes the classic
underfit -> sweet-spot -> overfit transition very clearly:

* `max_depth=3`: probably underfit (the tree can't represent the problem).
* `max_depth=10` with `min_samples_leaf` >> 1: usually the healthiest setting.
* `max_depth=None`: textbook overfit (train AUC ~ 1.0, val AUC much lower).

We also try the `entropy` criterion to confirm it gives essentially identical
results to `gini` for this dataset.


In [ ]:
def fit_eval(model, X_tr, y_tr, X_val, y_val, cv=None):
    """Train + return metrics dict (no MLflow side-effects)."""
    model.fit(X_tr, y_tr)
    if hasattr(model, "predict_proba"):
        p_tr  = model.predict_proba(X_tr)[:, 1]
        p_val = model.predict_proba(X_val)[:, 1]
    else:
        p_tr  = model.decision_function(X_tr)
        p_val = model.decision_function(X_val)
    pr_tr  = (p_tr  > 0.5).astype(int)
    pr_val = (p_val > 0.5).astype(int)
    metrics = {
        'train_auc'   : roc_auc_score(y_tr,  p_tr),
        'val_auc'     : roc_auc_score(y_val, p_val),
        'train_ap'    : average_precision_score(y_tr,  p_tr),
        'val_ap'      : average_precision_score(y_val, p_val),
        'val_f1'      : f1_score(y_val, pr_val, zero_division=0),
        'val_prec'    : precision_score(y_val, pr_val, zero_division=0),
        'val_recall'  : recall_score(y_val, pr_val, zero_division=0),
        'overfit_gap' : roc_auc_score(y_tr, p_tr) - roc_auc_score(y_val, p_val),
    }
    if cv is not None:
        cv_aucs = cross_val_score(model, X_tr, y_tr, cv=cv,
                                  scoring='roc_auc', n_jobs=-1)
        metrics['cv_auc_mean'] = cv_aucs.mean()
        metrics['cv_auc_std']  = cv_aucs.std()
    return metrics

def print_m(tag, m):
    print(f"  [{tag}]")
    for k, v in m.items():
        print(f"    {k:14s} = {v:.5f}")

results_log = []


In [ ]:
from sklearn.tree import DecisionTreeClassifier

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
X_sel = X_train_fe[SELECTED_FEATURES].astype(np.float32).values
X_tr, X_val, y_tr, y_val = train_test_split(X_sel, y, test_size=0.2,
                                            stratify=y, random_state=SEED)
print(f"X_tr {X_tr.shape}, X_val {X_val.shape}")


In [ ]:
m = DecisionTreeClassifier(max_depth=3,  random_state=SEED, class_weight='balanced')
mt = fit_eval(m, X_tr, y_tr, X_val, y_val)
print_m("depth=3 underfit", mt)
results_log.append({'name': "DT_depth=3 underfit", 'model': m, **mt})


In [ ]:
m = DecisionTreeClassifier(max_depth=5,  random_state=SEED, class_weight='balanced')
mt = fit_eval(m, X_tr, y_tr, X_val, y_val)
print_m("depth=5", mt)
results_log.append({'name': "DT_depth=5", 'model': m, **mt})


In [ ]:
m = DecisionTreeClassifier(max_depth=10, random_state=SEED, class_weight='balanced')
mt = fit_eval(m, X_tr, y_tr, X_val, y_val)
print_m("depth=10", mt)
results_log.append({'name': "DT_depth=10", 'model': m, **mt})


In [ ]:
m = DecisionTreeClassifier(max_depth=15, random_state=SEED, class_weight='balanced')
mt = fit_eval(m, X_tr, y_tr, X_val, y_val)
print_m("depth=15", mt)
results_log.append({'name': "DT_depth=15", 'model': m, **mt})


In [ ]:
m = DecisionTreeClassifier(max_depth=None, random_state=SEED, class_weight='balanced')
mt = fit_eval(m, X_tr, y_tr, X_val, y_val)
print_m("depth=None overfit", mt)
results_log.append({'name': "DT_depth=None overfit", 'model': m, **mt})


In [ ]:
m = DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, random_state=SEED, class_weight='balanced')
mt = fit_eval(m, X_tr, y_tr, X_val, y_val)
print_m("depth=10 + min_leaf=20", mt)
results_log.append({'name': "DT_depth=10 + min_leaf=20", 'model': m, **mt})


In [ ]:
m = DecisionTreeClassifier(max_depth=10, min_samples_split=50, random_state=SEED, class_weight='balanced')
mt = fit_eval(m, X_tr, y_tr, X_val, y_val)
print_m("depth=10 + min_split=50", mt)
results_log.append({'name': "DT_depth=10 + min_split=50", 'model': m, **mt})


In [ ]:
m = DecisionTreeClassifier(max_depth=10, criterion='entropy', random_state=SEED, class_weight='balanced')
mt = fit_eval(m, X_tr, y_tr, X_val, y_val)
print_m("entropy criterion", mt)
results_log.append({'name': "DT_entropy criterion", 'model': m, **mt})


In [ ]:
df_results = pd.DataFrame([{k:v for k,v in r.items() if k!='model'} for r in results_log])
df_results = df_results.sort_values('val_auc', ascending=False).reset_index(drop=True)

# Diagnose each run: overfit / underfit / healthy.
# Heuristic for fraud detection (AUC):
#   - train AUC < 0.75  -> underfit (model can't learn the task)
#   - overfit_gap > 0.05 -> overfit (memorising train, fails on val)
#   - 0 <= overfit_gap <= 0.02 and val_auc > 0.85 -> healthy
def _diag(row):
    if row['train_auc'] < 0.75:
        return 'UNDERFIT'
    if row['overfit_gap'] > 0.05:
        return 'OVERFIT'
    if row['overfit_gap'] < 0:
        return 'lucky-val'
    if row['val_auc'] >= 0.85 and row['overfit_gap'] <= 0.02:
        return 'HEALTHY'
    return 'mild-overfit' if row['overfit_gap'] > 0.02 else 'ok'
df_results['diagnosis'] = df_results.apply(_diag, axis=1)

_default_cols = ['name','train_auc','val_auc','val_f1','val_ap','overfit_gap','diagnosis']
show_cols = [c for c in _default_cols if c in df_results.columns]
print(df_results[show_cols].to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
df_results.set_index('name')[['train_auc','val_auc']].plot(kind='bar', ax=ax[0])
ax[0].set_title('DecisionTree: Train vs Val AUC')
ax[0].set_ylim(max(0.5, df_results[['train_auc','val_auc']].min().min()-0.05), 1.0)
ax[0].tick_params(axis='x', rotation=30); ax[0].legend(loc='lower right')

colors = ['salmon' if g > 0.05 else ('orange' if g > 0.02 else 'seagreen')
          for g in df_results['overfit_gap']]
df_results.set_index('name')['overfit_gap'].plot(kind='bar', ax=ax[1], color=colors)
ax[1].axhline(0,    color='black', lw=0.5)
ax[1].axhline(0.02, color='orange', ls='--', lw=0.5, label='mild')
ax[1].axhline(0.05, color='red',    ls='--', lw=0.5, label='overfit')
ax[1].set_title('DecisionTree: overfit gap (Train AUC - Val AUC)')
ax[1].tick_params(axis='x', rotation=30); ax[1].legend(loc='upper right')
plt.tight_layout(); plt.show()

n_over   = int((df_results['diagnosis']=='OVERFIT').sum())
n_under  = int((df_results['diagnosis']=='UNDERFIT').sum())
n_health = int((df_results['diagnosis']=='HEALTHY').sum())
best     = df_results.iloc[0]
worst    = df_results.iloc[-1]

print(f"""
=========== ANALYSIS (DecisionTree) ===========
- {len(df_results)} runs total | HEALTHY: {n_health} | OVERFIT: {n_over} | UNDERFIT: {n_under}
- Best   : {best['name']}  ->  val AUC = {best['val_auc']:.5f},  gap = {best['overfit_gap']:+.4f}
- Worst  : {worst['name']}  ->  val AUC = {worst['val_auc']:.5f}, gap = {worst['overfit_gap']:+.4f}

How to read the diagnosis column:
- overfit_gap > 0.05  -> OVERFIT  (train AUC is much higher than val AUC, the model
                                   memorized the training data)
- overfit_gap <= 0.02 -> HEALTHY  (model learned the signal and generalizes)
- train_auc  < 0.75   -> UNDERFIT (high bias, model is too simple OR regularization
                                   is too strong)
- For fraud detection, recall (true-positive rate on the fraud class) is often
  prioritized alongside AUC. If the business wants high recall, the decision
  threshold should be moved below 0.5 - the AUC ranking does not change but the
  precision/recall trade-off does.
""")
best_model = [r['model'] for r in results_log if r['name']==best['name']][0]
print(f"-> picked best_model = {best['name']}")


## 5. Pipeline Construction & Save

We bundle the whole preprocessing chain and the best trained model into one
sklearn `Pipeline` that can be applied **directly to the raw test CSVs** -
no separate preprocessing step required at inference time. The pipeline is
saved both as a local pickle and as an MLflow artifact (next section).


In [ ]:
import pickle
from sklearn.pipeline import Pipeline

class ColumnSelector(BaseEstimator, TransformerMixin):
    def __init__(self, cols): self.cols = cols
    def fit(self, X, y=None): return self
    def transform(self, X): return X[self.cols]

# Build a single pipeline that runs on RAW test data
final_pipeline = Pipeline([
    ('feat',   FeatureEngineer()),
    ('catenc', CategoricalEncoder()),
    ('impute', Imputer()),
    ('select', ColumnSelector(SELECTED_FEATURES)),
    ('model',  best_model),
])

# Refit FE part + best model on the FULL training data (raw)
final_pipeline.fit(X_train_raw, y)
print("Pipeline fitted on full raw training data.")

# Sanity: probabilistic predictions on raw test
test_pred_proba = final_pipeline.predict_proba(X_test_raw)[:, 1]
print(f"Test prediction probabilities sample: {test_pred_proba[:5]}")
print(f"Mean predicted P(fraud) on test set : {test_pred_proba.mean():.4f}")

# Save pipeline locally too (optional)
PIPE_PATH = f"pipeline_{MODEL_TAG}.pkl"
with open(PIPE_PATH, 'wb') as f:
    pickle.dump(final_pipeline, f)
print(f"Pipeline saved to {PIPE_PATH}")


## 6. MLflow Logging (run separately!)

This section is split into separate cells so you can finish all the modelling
work first and only push runs to MLflow when you are ready.

Experiment name: **DecisionTree_Training**.

The runs created here are:

* `<Model>_Cleaning` - dropped columns + before/after shapes
* `<Model>_Feature_Selection` - per-method AUC + chosen feature set
* `<Model>_<config>` - one run per hyperparameter combination from section 4
* `<Model>_CrossValidation` - 5-fold CV for the best config
* `<Model>_Final_Pipeline` - logs the complete sklearn `Pipeline` and registers
  it in the Model Registry under `IEEE_Fraud_<Model>`.


In [ ]:
# 6.3  Cleaning summary run
with mlflow.start_run(run_name=f"{MODEL_TAG}_Cleaning"):
    mlflow.log_param('stage', 'cleaning')
    mlflow.log_param('dropped_columns', len(DROP_COLS))
    mlflow.log_param('train_shape_after', str(train.shape))
    mlflow.log_param('test_shape_after',  str(test.shape))
print("Cleaning run logged.")


In [ ]:
# 6.2  Feature-Selection comparison run
with mlflow.start_run(run_name=f"{MODEL_TAG}_Feature_Selection"):
    mlflow.log_param('stage', 'feature_selection')
    mlflow.log_param('chosen', best_fs_name if 'best_fs_name' in dir() else 'n/a')
    mlflow.log_param('n_selected', len(SELECTED_FEATURES))
    if 'fs_df' in dir():
        for _, row in fs_df.iterrows():
            mlflow.log_metric(f"AUC_{row['method']}", float(row['mean_auc']))
print("Feature Selection run logged.")


In [ ]:
# 6.1  Per-hyperparameter runs
mlflow.set_experiment(MLFLOW_EXPERIMENT)

for r in results_log:
    with mlflow.start_run(run_name=r['name']):
        # Params (model + general)
        mlflow.log_param('model_type',        MODEL_TAG)
        mlflow.log_param('n_features',        len(SELECTED_FEATURES))
        mlflow.log_param('feature_selection', best_fs_name if 'best_fs_name' in dir() else 'manual')
        mlflow.log_param('config',            r['name'])
        # Metrics
        for k, v in r.items():
            if k in ('name','model'): continue
            try: mlflow.log_metric(k, float(v))
            except Exception: pass
print("Logged all training runs to MLflow.")


In [ ]:
# 6.4  Cross-validation run for the BEST hyperparameter set
print("Re-running 5-fold CV for the BEST config (this can take a few min)...")
cv5_aucs = cross_val_score(best_model, X_train_fe[SELECTED_FEATURES].values, y,
                           cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                           scoring='roc_auc', n_jobs=-1)
print(f"CV AUC mean = {cv5_aucs.mean():.5f} +/- {cv5_aucs.std():.5f}")

with mlflow.start_run(run_name=f"{MODEL_TAG}_CrossValidation"):
    mlflow.log_param('stage', 'cross_validation')
    mlflow.log_param('cv_folds', 5)
    mlflow.log_param('best_config', best['name'])
    mlflow.log_metric('cv_auc_mean', float(cv5_aucs.mean()))
    mlflow.log_metric('cv_auc_std',  float(cv5_aucs.std()))
    for i, a in enumerate(cv5_aucs):
        mlflow.log_metric(f'cv_auc_fold{i+1}', float(a))
print("Cross-validation run logged.")


In [ ]:
# 6.5  FINAL run -- log the trained Pipeline as MLflow artifact
with mlflow.start_run(run_name=f"{MODEL_TAG}_Final_Pipeline"):
    mlflow.log_param('model_type',        MODEL_TAG)
    mlflow.log_param('best_config',       best['name'])
    mlflow.log_param('n_features',        len(SELECTED_FEATURES))
    mlflow.log_param('feature_selection', best_fs_name if 'best_fs_name' in dir() else 'manual')

    mlflow.log_metric('best_val_auc',     float(best['val_auc']))
    mlflow.log_metric('best_train_auc',   float(best['train_auc']))
    mlflow.log_metric('best_overfit_gap', float(best['overfit_gap']))
    if 'cv5_aucs' in dir():
        mlflow.log_metric('cv_auc_mean', float(cv5_aucs.mean()))

    # Log entire pipeline (preprocessing + model) as an MLflow sklearn model
    # so model_inference can load it from the registry directly.
    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        artifact_path='model',
        registered_model_name=f'IEEE_Fraud_{MODEL_TAG}',
    )

print(f"Final pipeline logged & registered as 'IEEE_Fraud_{MODEL_TAG}'.")
print("In model_inference.ipynb you can now load it via:")
print(f"    mlflow.sklearn.load_model('models:/IEEE_Fraud_{MODEL_TAG}/latest')")
